In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# Folder containing resized images
image_folder = "data/resized"


def extract_glcm_features(img):
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Reduce intensity levels (0–255 → 0–15)
    gray = (gray / 16).astype(np.uint8)

    # Create GLCM matrix
    glcm = graycomatrix(
        gray,
        distances=[1],
        angles=[0],
        levels=16,
        symmetric=True,
        normed=True
    )

    # Extract features
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

    return [contrast, correlation, energy, homogeneity]


# Store results
features = []
image_ids = []

for file in os.listdir(image_folder):
    if file.endswith(".jpg"):
        path = os.path.join(image_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        feat = extract_glcm_features(img)

        features.append(feat)
        image_ids.append(file.replace(".jpg", ""))


# Create DataFrame
glcm_df = pd.DataFrame(features, columns=[
    "contrast",
    "correlation",
    "energy",
    "homogeneity"
])

glcm_df["image_id"] = image_ids

print(glcm_df.head())

# Save to CSV
glcm_df.to_csv("glcm_features.csv", index=False)

print("✅ GLCM features extracted and saved!")